# Solution R1 · Warm-up: the first regression, from data to number

**Type**: Implement with scikit-learn

## Objective

Go through the minimal cycle of a model on your own: split the data, train a **dumb baseline** (the mean), train a linear regression, compare the error on the training and test sets and read the coefficients.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X, y = ds.diabetes()
print(X.shape, "| target: mean", round(float(y.mean()), 1), "standard deviation", round(float(y.std()), 1))

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
baseline = DummyRegressor().fit(X_tr, y_tr)
baseline_r2 = baseline.score(X_te, y_te)
baseline_rmse = mean_squared_error(y_te, baseline.predict(X_te), squared=False)
print(f"baseline (the mean): test R² {baseline_r2:.3f} | RMSE {baseline_rmse:.1f}")

In [ ]:
model = LinearRegression().fit(X_tr, y_tr)
train_r2, test_r2 = model.score(X_tr, y_tr), model.score(X_te, y_te)
test_rmse = mean_squared_error(y_te, model.predict(X_te), squared=False)
print(f"linear regression: training R² {train_r2:.3f} | test R² {test_r2:.3f} | test RMSE {test_rmse:.1f}")

In [ ]:
coef = pd.Series(model.coef_, index=X.columns)
most_influential = coef.abs().idxmax()
print(coef.round(0).sort_values().to_string())
print("most influential:", most_influential)

## Check

In [ ]:
with verify():
    check(len(X_tr) == 331 and len(X_te) == 111, "75/25 split: 331 training rows and 111 test rows", "test_size=0.25, random_state=0")
    check(abs(baseline_r2) < 0.05, "the baseline has R² close to 0: predicting the mean explains nothing", "DummyRegressor().score(X_te, y_te)")
    check(0.3 < test_r2 < 0.5 and train_r2 > test_r2, "test R² between 0.3 and 0.5, and lower than the training one", "model.score(X_te, y_te)")
    check(test_rmse < baseline_rmse - 8, "the error of the model is at least 8 points below that of the baseline", "squared=False for the root")
    check(most_influential == "s5", "the variable with the largest coefficient is s5", "coef.abs().idxmax()")
done()

## What you learned

- **The baseline is worth R² ≈ 0:** always predicting the mean gives 0 (the test set has a slightly different mean, so 0.000 or a little less). A model is useful only if it beats this number: the linear regression reaches R² 0.36 and reduces the RMSE from 70.5 to 56.4.
- **Training and test do not coincide:** R² 0.56 on the training set and 0.36 on the test set. The gap is normal with 331 examples and 10 variables, and it is the reason the evaluation is always done on data the model has not seen.
- **Not all the coefficients are readable:** `s5` and `bmi` have the highest coefficients, but `s1` (−560) and `s2` (+261) are highly correlated with each other and their coefficients compensate each other: alone they mean nothing. You come back to this in module 1 (exercises C1 and D1).

## Common pitfall

Reporting the training R² as "the quality of the model": it is always optimistic. And reading an R² without comparing it with the baseline.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What does an R² of 0.36 on the test set mean?</summary>

That the model explains 36% of the variance of the target compared with always predicting the training mean (which is worth 0): it is useful but leaves a lot of unexplained variability. The number has to be read on the test set and next to the baseline, and accompanied by an error in the units of the target (RMSE).

</details>